# AM5061 · Week 4 · Steam header insulation

**Design of Thermal and Fluid Systems** · Applied Mechanics, IIT Madras · Jul–Nov 2026

Run the two setup cells below once, then work down the notebook. Nothing needs to be installed on your own machine.


## Setup

Run these two cells first. The second one writes the course helper module, so this notebook is self-contained.


In [ ]:
#@title Install the property library  { display-mode: "form" }
!pip install -q CoolProp openpyxl
print('CoolProp ready')


In [ ]:
%%writefile am5061.py
"""AM5061 - Design of Thermal and Fluid Systems.
Shared helpers for the course notebooks.

Design of Thermal and Fluid Systems, IIT Madras, Jul-Nov 2026.

This module is deliberately thin. It wraps CoolProp with names and units that
match the lecture notation, adds an Excel writer that produces workbooks you
can actually filter, and sets a consistent plot style. It does NOT hide the
engineering: every case notebook still writes its own equations.

Install (first cell of any Colab notebook):
    !pip install -q CoolProp openpyxl

Units are SI throughout, with ONE exception that is flagged everywhere it
appears: temperatures in function arguments named `..._C` are in Celsius,
because that is how the case briefs state them. Everything internal is kelvin.
"""
from __future__ import annotations

import math

from CoolProp.CoolProp import PropsSI, PhaseSI

__all__ = [
    "K", "C", "State", "state", "sat_liquid", "sat_vapour", "p_sat", "T_sat",
    "h_fg", "critical", "fluids", "solve", "sweep", "to_excel",
    "style_plots", "NAVY", "ORANGE", "BLUE", "MUTED",
]

# ---------------------------------------------------------------- constants
NAVY, ORANGE, BLUE, MUTED = "#1F3864", "#ED7D31", "#4472C4", "#59626E"
T0 = 273.15


def K(t_celsius: float) -> float:
    """Celsius -> kelvin. Use this at the boundary, never inside a formula."""
    return t_celsius + T0


def C(t_kelvin: float) -> float:
    """Kelvin -> Celsius, for reporting only."""
    return t_kelvin - T0


# ------------------------------------------------------------------- states
class State:
    """A thermodynamic state. Immutable, and it knows its own fluid.

    Construct it with any two independent properties:
        State("R134a", P=1e6, T=K(70))
        State("Water", P=101325, Q=0)      # saturated liquid
        State("R134a", P=p_cond, H=h2)

    Then read properties as attributes: .T .p .h .s .d .cp .x
    Attribute names match the lecture notation, not CoolProp's letter codes,
    so a student reading the notebook does not need the CoolProp manual open.
    """

    _MAP = {"T": "T", "P": "P", "H": "H", "S": "S", "D": "D", "Q": "Q"}

    def __init__(self, fluid: str, **kw):
        if len(kw) != 2:
            raise ValueError(
                f"a state needs exactly two properties, got {list(kw)}. "
                "Two and only two - that is the phase rule, not a quirk."
            )
        (n1, v1), (n2, v2) = kw.items()
        for n in (n1, n2):
            if n not in self._MAP:
                raise ValueError(f"unknown property {n!r}; use T, P, H, S, D or Q")
        self.fluid, self._args = fluid, (n1, v1, n2, v2)

    def _get(self, what: str) -> float:
        n1, v1, n2, v2 = self._args
        return PropsSI(what, n1, v1, n2, v2, self.fluid)

    # Named so they read like the equations on the slides.
    T  = property(lambda s: s._get("T"),  doc="temperature, K")
    p  = property(lambda s: s._get("P"),  doc="pressure, Pa")
    h  = property(lambda s: s._get("H"),  doc="specific enthalpy, J/kg")
    s  = property(lambda s: s._get("S"),  doc="specific entropy, J/kg.K")
    d  = property(lambda s: s._get("D"),  doc="density, kg/m3")
    cp = property(lambda s: s._get("C"),  doc="cp, J/kg.K")
    mu = property(lambda s: s._get("V"),  doc="dynamic viscosity, Pa.s")
    k  = property(lambda s: s._get("L"),  doc="thermal conductivity, W/m.K")
    x  = property(lambda s: s._get("Q"),  doc="vapour quality, - (=-1 if single phase)")

    @property
    def T_C(self) -> float:
        return C(self.T)

    @property
    def phase(self) -> str:
        n1, v1, n2, v2 = self._args
        return PhaseSI(n1, v1, n2, v2, self.fluid)

    def __repr__(self):
        try:
            return (f"State({self.fluid}: {self.T_C:.2f} C, {self.p/1e5:.3f} bar, "
                    f"h={self.h/1e3:.2f} kJ/kg, {self.phase})")
        except Exception:
            return f"State({self.fluid}, {self._args})"


def state(fluid: str, **kw) -> State:
    """Shorthand for State(...)."""
    return State(fluid, **kw)


def sat_liquid(fluid: str, *, T=None, p=None) -> State:
    """Saturated liquid at T or p. Give one, not both."""
    if (T is None) == (p is None):
        raise ValueError("give exactly one of T or p")
    return State(fluid, T=T, Q=0) if T is not None else State(fluid, P=p, Q=0)


def sat_vapour(fluid: str, *, T=None, p=None) -> State:
    """Saturated vapour at T or p."""
    if (T is None) == (p is None):
        raise ValueError("give exactly one of T or p")
    return State(fluid, T=T, Q=1) if T is not None else State(fluid, P=p, Q=1)


def p_sat(fluid: str, T: float) -> float:
    """Saturation pressure, Pa. For a BLEND this is the bubble-point pressure."""
    return PropsSI("P", "T", T, "Q", 0, fluid)


def T_sat(fluid: str, p: float) -> float:
    """Saturation temperature, K.

    WARNING for blends: a zeotropic mixture has no single saturation
    temperature. This returns the BUBBLE point. Use glide() to see the spread.
    """
    return PropsSI("T", "P", p, "Q", 0, fluid)


def glide(fluid: str, p: float) -> float:
    """Dew minus bubble temperature at p, K. Zero for a pure fluid."""
    return (PropsSI("T", "P", p, "Q", 1, fluid)
            - PropsSI("T", "P", p, "Q", 0, fluid))


def h_fg(fluid: str, *, T=None, p=None) -> float:
    """Latent heat, J/kg."""
    return sat_vapour(fluid, T=T, p=p).h - sat_liquid(fluid, T=T, p=p).h


def critical(fluid: str) -> dict:
    """Critical point, for checking you are not extrapolating past it."""
    return {"T": PropsSI("TCRIT", fluid), "p": PropsSI("PCRIT", fluid)}


def fluids() -> list:
    """Every fluid CoolProp knows. There are about 130."""
    import CoolProp
    return sorted(CoolProp.__fluids__)


# ------------------------------------------------------------------ solvers
def solve(f, x0, *, tol=1e-10, max_iter=200, bracket=None):
    """Find x where f(x) = 0.

    Uses Brent's method when you give a bracket (robust, always converges if
    the bracket is valid), otherwise secant from x0. Raises with a readable
    message rather than returning a wrong answer silently, which is the whole
    problem with doing this in a spreadsheet.
    """
    from scipy.optimize import brentq, newton
    if bracket is not None:
        a, b = bracket
        fa, fb = f(a), f(b)
        if fa * fb > 0:
            raise ValueError(
                f"f({a:g})={fa:g} and f({b:g})={fb:g} have the same sign, so no "
                "root is bracketed. Widen the bracket or check the equation."
            )
        return brentq(f, a, b, xtol=tol, maxiter=max_iter)
    return newton(f, x0, tol=tol, maxiter=max_iter)


def sweep(fn, values, *, name="x"):
    """Run fn(v) for each v and collect the results as a list of dicts.

    fn must return a dict. The sweep variable is added under `name`, so the
    result drops straight into to_excel().
    """
    rows = []
    for v in values:
        out = fn(v)
        if not isinstance(out, dict):
            raise TypeError("the swept function must return a dict of results")
        rows.append({name: v, **out})
    return rows


# -------------------------------------------------------------------- excel
def to_excel(path, sheets: dict, *, sources=None, summary=None, title=None):
    """Write a workbook that is genuinely usable.

    Every numeric cell is written as a number (not text), AutoFilter is on,
    the header row is frozen, and columns are sized to content. A Summary and
    a Sources sheet are always present, because a result you cannot trace is
    not an engineering deliverable.

        sheets  = {"Sweep": [ {...}, {...} ], ...}   list of dicts per sheet
        sources = [ ("what", "where it came from"), ... ]
        summary = [ ("quantity", value, "units"), ... ]
    """
    from openpyxl import Workbook
    from openpyxl.styles import Font, PatternFill, Alignment
    from openpyxl.utils import get_column_letter

    wb = Workbook()
    wb.remove(wb.active)
    head_font = Font(bold=True, color="FFFFFF", name="Calibri")
    head_fill = PatternFill("solid", fgColor="1F3864")

    def _write(ws, rows, headers=None):
        headers = headers or (list(rows[0].keys()) if rows else [])
        for j, hname in enumerate(headers, 1):
            c = ws.cell(row=1, column=j, value=hname)
            c.font, c.fill = head_font, head_fill
            c.alignment = Alignment(horizontal="left")
        for i, row in enumerate(rows, 2):
            for j, hname in enumerate(headers, 1):
                v = row.get(hname)
                # Numbers stay numbers. This is the single most common way a
                # delivered workbook turns out not to be filterable.
                if isinstance(v, bool):
                    v = str(v)
                elif isinstance(v, (int, float)) and not isinstance(v, bool):
                    v = float(v) if isinstance(v, float) else v
                ws.cell(row=i, column=j, value=v)
        if rows:
            ws.auto_filter.ref = (f"A1:{get_column_letter(len(headers))}"
                                  f"{len(rows) + 1}")
        ws.freeze_panes = "A2"
        for j, hname in enumerate(headers, 1):
            width = max([len(str(hname))] +
                        [len(f"{r.get(hname)}") for r in rows[:200]]) + 3
            ws.column_dimensions[get_column_letter(j)].width = min(width, 42)

    # Summary first, so it is what opens.
    ws = wb.create_sheet("Summary")
    ws["A1"] = title or "AM5061 results"
    ws["A1"].font = Font(bold=True, size=14, color="1F3864")
    r = 3
    for item in (summary or []):
        for j, v in enumerate(item, 1):
            ws.cell(row=r, column=j, value=v)
        r += 1
    ws.column_dimensions["A"].width = 46
    ws.column_dimensions["B"].width = 18
    ws.column_dimensions["C"].width = 14

    for sname, rows in sheets.items():
        _write(wb.create_sheet(sname[:31]), rows)

    ws = wb.create_sheet("Sources")
    _write(ws, [{"item": a, "source": b} for a, b in (sources or [])])

    wb.save(path)
    return path


# --------------------------------------------------------------- plot style
def style_plots():
    """Match the lecture decks, so figures in a report look like the slides."""
    import matplotlib as mpl
    mpl.rcParams.update({
        "figure.figsize": (7.2, 4.4), "figure.dpi": 110,
        "axes.edgecolor": MUTED, "axes.labelcolor": NAVY,
        "axes.titlecolor": NAVY, "axes.titlesize": 11.5,
        "axes.spines.top": False, "axes.spines.right": False,
        "axes.grid": True, "grid.alpha": 0.25, "grid.linewidth": 0.6,
        "xtick.color": MUTED, "ytick.color": MUTED,
        "font.size": 10, "legend.frameon": False,
        "axes.prop_cycle": mpl.cycler(color=[NAVY, ORANGE, BLUE, "#7F9DB9"]),
    })


---
## The case

A **DN 150 steam header** at a Tiruppur textile mill, carrying saturated steam
at **8 bar(a)**. The energy manager wants the heat loss per metre at several
insulation thicknesses, and then asks a second question that sounds naive and
is not: *if insulation reduces loss, does more insulation always reduce it
further?*

Deliverable **D-4**: loss per metre at t = 25, 50, 75, 100, 150 mm, plus the
surface temperature and the conductivity the insulation actually ended up at.

### Three mechanisms, and one of them is a loop

Conduction through steel and wool, natural convection off the cladding, and
radiation from it. The wool's conductivity **rises with temperature**, so its
resistance depends on the answer. That is a loop, and you close it by iterating.


## 1. The resistance network

Four resistances in series. The cylindrical conduction term is the one people
get wrong: because area grows with radius, the integral of `dr/(2πrkL)` gives a
**logarithm**, not `thickness/(k·A)`. On thick insulation that error is worth
tens of percent.


In [ ]:
import am5061 as am
import numpy as np, matplotlib.pyplot as plt
from CoolProp.CoolProp import PropsSI
from scipy.optimize import brentq
am.style_plots()

SIGMA, G_N, PI = 5.670374419e-8, 9.80665, np.pi

# geometry and boundary conditions, from the brief
r1, r2   = 0.077025, 0.08415      # m   DN 150 Sch 40 inside / outside radius
T_steam  = 443.564                # K   saturated steam at 8 bar(a)
T_amb    = 308.15                 # K   35 C shop floor
h_steam  = 8000.0                 # W/m2K  condensing/flowing steam film
eps_clad = 0.15                   # -   bright aluminium cladding
k_steel  = 50.0                   # W/mK
k0_ins, b_ins = 0.033, 1.5e-4     # k_ins = k0 + b*(T_mean_C)

def h_churchill_chu(T_s, T_amb, D):
    """Natural convection, horizontal cylinder. Valid to Ra ~ 1e12."""
    T_f = min(max(0.5*(T_s + T_amb), 250), 800)     # film temperature
    rho = PropsSI('D','P',101325,'T',T_f,'Air'); mu = PropsSI('V','P',101325,'T',T_f,'Air')
    k   = PropsSI('L','P',101325,'T',T_f,'Air'); cp = PropsSI('C','P',101325,'T',T_f,'Air')
    Pr, nu, alpha, beta = cp*mu/k, mu/rho, k/(rho*cp), 1/T_f
    Ra = max(G_N*beta*(T_s - T_amb)*D**3/(nu*alpha), 1e-8)
    Nu = (0.60 + 0.387*Ra**(1/6)/(1 + (0.559/Pr)**(9/16))**(8/27))**2
    return Nu*k/D

def h_radiation(T_s, T_sur, eps=eps_clad):
    """Linearised, and this is EXACT, not an approximation: Ts^4 - Tsur^4
    factorises into (Ts^2+Tsur^2)(Ts+Tsur)(Ts-Tsur)."""
    return eps*SIGMA*(T_s**2 + T_sur**2)*(T_s + T_sur)


## 2. Solving the loop

Two unknowns are tangled: the surface temperature sets the outside coefficient,
and the mean insulation temperature sets its conductivity. Guess the heat flow,
work out both, check the network closes, iterate.


In [ ]:
def header(t_ins=0.050, L=1.0):
    r3 = r2 + t_ins
    A  = 2*PI*r3*L
    R_steam = 1/(h_steam*2*PI*r1*L)
    R_steel = np.log(r2/r1)/(2*PI*k_steel*L)

    h_tot = lambda Ts: h_churchill_chu(Ts, T_amb, 2*r3) + h_radiation(Ts, T_amb)
    def surface_T(q):
        f = lambda Ts: h_tot(Ts)*A*(Ts - T_amb) - q
        hi = T_steam
        while f(hi) < 0 and hi < 2000.0:   # cannot shed q even at steam temp
            hi += 200.0
        return brentq(f, T_amb + 1e-9, hi)

    def residual(q):
        T2 = T_steam - q*(R_steam + R_steel)        # insulation inner face
        Ts = surface_T(q)
        k_eff = max(k0_ins + b_ins*(0.5*(T2+Ts) - 273.15), 0.005)
        R_ins = np.log(r3/r2)/(2*PI*k_eff*L)
        return q*(R_steam + R_steel + R_ins) - (T_steam - Ts)

    # The heat flow varies by an order of magnitude across the thickness
    # sweep, so a fixed bracket does not hold. Expand until the sign changes.
    lo, hi = 0.1, 200.0
    while residual(lo)*residual(hi) > 0 and hi < 1e5:
        hi *= 2.0
    q  = brentq(residual, lo, hi)
    Ts = surface_T(q)
    T2 = T_steam - q*(R_steam + R_steel)
    k_eff = max(k0_ins + b_ins*(0.5*(T2+Ts) - 273.15), 0.005)
    R_ins = np.log(r3/r2)/(2*PI*k_eff*L)
    R_out = 1/(h_tot(Ts)*A)
    return {"t_ins, mm": t_ins*1e3, "q, W/m": q, "T_surface, C": Ts - 273.15,
            "k_ins used, W/mK": k_eff, "h_out, W/m2K": h_tot(Ts),
            "R_steam": R_steam, "R_steel": R_steel, "R_ins": R_ins, "R_out": R_out,
            "r_critical, mm": k_eff/h_tot(Ts)*1e3, "r3, mm": r3*1e3}

base = header(0.050)
for k, v in base.items():
    print(f"  {k:20s} {v:12.5f}")


> **Check.** At 50 mm the loss is **78.24 W/m** with a surface at
> about 53.7 °C.
>
> Note where the resistance sits: the insulation carries almost all of it. The
> steam film and the steel wall are together worth well under a percent, which
> is why nobody specifies a steam-side coefficient carefully on a job like this.


## 3. The deliverable table

In [ ]:
rows = [header(t) for t in (0.025, 0.050, 0.075, 0.100, 0.150)]
print(f"{'t, mm':>7}{'q, W/m':>10}{'T_surf, C':>11}{'k_ins':>9}{'h_out':>8}{'R_ins %':>9}")
for r_ in rows:
    frac = 100*r_["R_ins"]/(r_["R_steam"]+r_["R_steel"]+r_["R_ins"]+r_["R_out"])
    print(f"{r_['t_ins, mm']:7.0f}{r_['q, W/m']:10.3f}{r_['T_surface, C']:11.2f}"
          f"{r_['k_ins used, W/mK']:9.5f}{r_['h_out, W/m2K']:8.3f}{frac:9.1f}")
print("\n  Note the diminishing return: doubling 25 -> 50 mm saves far more"
      "\n  than doubling 75 -> 150 mm. The log is why.")


## 4. The second question: the critical radius

Add insulation and two things happen at once. The conduction path gets
**longer**, which helps. The outside surface gets **bigger**, which hurts.
Differentiate the total resistance and set it to zero:

`R_tot = ln(r/r_b)/(2πk) + 1/(2πrh)`  →  `dR/dr = 1/(2πkr) − 1/(2πhr²) = 0`  →  **r_cr = k/h**

Below that radius, adding insulation **raises** the loss.


In [ ]:
def bare_cylinder(t, r_bare, k_ins, h_out, T_core, T_amb=T_amb):
    r = r_bare + t
    R = np.log(r/r_bare)/(2*PI*k_ins) + 1/(2*PI*r*h_out)
    return (T_core - T_amb)/R

t = np.linspace(1e-6, 0.040, 2000)

# a 3 mm PVC-insulated cable in still air
q_cable = np.array([bare_cylinder(x, 0.0015, 0.17, 10.0, 353.15) for x in t])
r_cr_cable = 0.17/10.0

# the mill header, using the coefficients it actually settled at
q_pipe = np.array([bare_cylinder(x, r2, base["k_ins used, W/mK"],
                                 base["h_out, W/m2K"], T_steam) for x in t])
r_cr_pipe = base["k_ins used, W/mK"]/base["h_out, W/m2K"]

i = int(np.argmax(q_cable))
print(f"  cable: peak {q_cable[i]:.3f} W/m at r = {(0.0015+t[i])*1e3:.2f} mm"
      f"   (r_cr = k/h = {r_cr_cable*1e3:.1f} mm)")
print(f"  pipe : r_cr = {r_cr_pipe*1e3:.2f} mm, but the bare radius is already "
      f"{r2*1e3:.1f} mm")
print(f"  ratio r_cr/r_bare -> cable {r_cr_cable/0.0015:.1f}, pipe {r_cr_pipe/r2:.3f}")
print("\n  Above 1, insulation can make things worse. Below 1, it never can.")


In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11.6, 4.3))
a1.plot((0.0015+t)*1e3, q_cable, color=am.ORANGE, lw=2.4)
a1.axvline(r_cr_cable*1e3, color=am.MUTED, ls="--")
a1.plot((0.0015+t[i])*1e3, q_cable[i], "o", ms=9, color=am.NAVY, zorder=5)
a1.text(r_cr_cable*1e3, q_cable.min(), "  r_cr = k/h", color=am.MUTED, fontsize=9)
a1.set_xlabel("outer radius  (mm)"); a1.set_ylabel("loss  (W/m)")
a1.set_title("3 mm cable: loss RISES to a peak at 17 mm")

a2.plot((r2+t)*1e3, q_pipe, color=am.NAVY, lw=2.4)
a2.set_xlabel("outer radius  (mm)"); a2.set_ylabel("loss  (W/m)")
a2.set_title("DN 150 header: falls from the first millimetre")
plt.tight_layout(); plt.show()


Same equation, opposite conclusion. The only difference is the
bare radius. This is why "always insulate more" is wrong as a rule and right as
a habit for pipes.


## 5. The workbook

In [ ]:
ts = np.arange(0.020, 0.201, 0.010)
sweep_rows = [header(float(x)) for x in ts]
crit_rows = [{"outer radius, mm": (0.0015+x)*1e3, "cable q, W/m": bare_cylinder(x,0.0015,0.17,10.,353.15)}
             for x in np.linspace(1e-6, 0.040, 81)]

path = am.to_excel("AM5061_D4_Insulation.xlsx",
    {"Thickness sweep": sweep_rows, "Critical radius (cable)": crit_rows},
    title="AM5061 D-4 . Tiruppur steam header insulation",
    summary=[("Steam temperature", T_steam-273.15, "C"),
             ("Ambient", T_amb-273.15, "C"),
             ("Pipe outside radius", r2*1e3, "mm"),
             ("Cladding emissivity", eps_clad, "-"),
             ("Loss at 50 mm", base["q, W/m"], "W/m"),
             ("Surface temperature at 50 mm", base["T_surface, C"], "C"),
             ("Critical radius, pipe", base["r_critical, mm"], "mm"),
             ("Critical radius, 3 mm cable", r_cr_cable*1e3, "mm")],
    sources=[("Air properties", "CoolProp 'Air' at film temperature, 101325 Pa"),
             ("Natural convection", "Churchill & Chu, horizontal cylinder"),
             ("Radiation", "grey body, linearised exactly"),
             ("Insulation k(T)", "k = 0.033 + 1.5e-4*T_mean_C, mineral wool"),
             ("Geometry", "DN 150 Sch 40, AM5061 brief D-4")])
print("written:", path)


## What to hand in

1. The thickness table: loss, surface temperature, and the conductivity the
   wool actually reached.
2. A recommended thickness **with a stated reason** — surface-temperature
   safety limit, payback, or both. Say which.
3. The critical-radius answer to the energy manager's second question, with the
   cable and the pipe on the same argument.
4. The workbook.

Note the surface temperatures. Anything above about 60 °C is a contact-burn
risk, and that may bind before economics does.
